# Zynee ML Training & Testing Plots (Inline)

This notebook renders charts inline under code cells (like your screenshot), and also saves PNGs to `ml-report/charts/inline-notebook`.


## Instant Preview (No Run Needed)\n
If inline outputs are empty, these embedded images show your generated plots immediately.\n
\n
### Model Evaluation PNGs\n
![Suicide CM](charts/inline-notebook/suicide_confusion_matrix.png)\n
![Suicide ROC](charts/inline-notebook/suicide_roc_curve.png)\n
![Suicide PR](charts/inline-notebook/suicide_precision_recall_curve.png)\n
![Suicide Prob Hist](charts/inline-notebook/suicide_probability_histogram.png)\n
![Journal Emotion CM](charts/inline-notebook/journal_emotion_confusion_matrix.png)\n
![Journal Sentiment CM](charts/inline-notebook/journal_sentiment_confusion_matrix.png)\n
![Model Accuracy F1](charts/inline-notebook/model_accuracy_f1_comparison.png)\n
\n
### Train/Test Dataset PNGs\n
![Mood Forecast Train Test](charts/inline-notebook/mood_forecast_train_test_distribution.png)\n
![Mood Pattern Train Test](charts/inline-notebook/mood_pattern_train_test_distribution.png)\n
![Weekly Wellness Train Test](charts/inline-notebook/weekly_wellness_train_test_distribution.png)\n
![Stats Box Hist](charts/inline-notebook/stats_box_train_test_histograms.png)\n
![Cosmic Insights Train Test](charts/inline-notebook/cosmic_insights_train_test_distribution.png)\n
![Journal Emotion Train Test](charts/inline-notebook/journal_emotion_train_test_distribution.png)\n
![Journal Sentiment Train Test](charts/inline-notebook/journal_sentiment_train_test_distribution.png)\n
![All Dataset Row Counts](charts/inline-notebook/all_datasets_train_test_row_counts.png)\n

In [ ]:
%matplotlib inline

from pathlib import Path
from collections import Counter
import csv

import numpy as np
import joblib
import matplotlib.pyplot as plt
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    confusion_matrix,
    roc_curve,
    auc,
    precision_recall_curve,
)

plt.style.use("seaborn-v0_8-whitegrid")

BASE_DIR = Path("/Users/angelajustin/Desktop/netset/zynee/zynee/assistant-microservice")
TRAIN_DIR = BASE_DIR / "data" / "training"
TEST_DIR = BASE_DIR / "data" / "testing"
PROCESSED_DIR = BASE_DIR / "data" / "processed"
MODELS_DIR = BASE_DIR / "models"
SAVE_DIR = BASE_DIR / "ml-report" / "charts" / "inline-notebook"
SAVE_DIR.mkdir(parents=True, exist_ok=True)

print(f"Base: {BASE_DIR}")
print(f"Saving PNGs to: {SAVE_DIR}")

def load_model(path: Path):
    loaded = joblib.load(path)
    if isinstance(loaded, dict) and "pipeline" in loaded:
        metadata = loaded.get("metadata", {})
        if not isinstance(metadata, dict):
            metadata = {}
        return loaded["pipeline"], metadata
    return loaded, {}

def read_text_label_csv(path: Path):
    texts, labels = [], []
    with path.open("r", encoding="utf-8", newline="", errors="ignore") as f:
        reader = csv.DictReader(f)
        for row in reader:
            text = str(row.get("text", "")).strip()
            label = str(row.get("label", "")).strip()
            if text and label:
                texts.append(text)
                labels.append(label)
    return texts, labels

def normalize_suicide_label(value: str) -> int:
    v = value.strip().lower()
    if v in {"suicide", "suicidal", "1", "true", "yes", "positive"}:
        return 1
    if v in {"non-suicide", "non suicide", "0", "false", "no", "negative", "normal"}:
        return 0
    raise ValueError(f"Unexpected suicide label: {value}")

def read_col(path: Path, column: str):
    values = []
    with path.open("r", encoding="utf-8", newline="", errors="ignore") as f:
        reader = csv.DictReader(f)
        for row in reader:
            val = str(row.get(column, "")).strip()
            if val:
                values.append(val)
    return values

def read_numeric_col(path: Path, column: str):
    values = []
    with path.open("r", encoding="utf-8", newline="", errors="ignore") as f:
        reader = csv.DictReader(f)
        for row in reader:
            raw = str(row.get(column, "")).strip()
            if not raw:
                continue
            try:
                values.append(float(raw))
            except ValueError:
                pass
    return values

def draw_cm(ax, cm, labels, title):
    im = ax.imshow(cm, interpolation="nearest", cmap="Blues")
    ax.set_title(title)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    ax.set_xticks(np.arange(len(labels)))
    ax.set_yticks(np.arange(len(labels)))
    ax.set_xticklabels(labels, rotation=25, ha="right")
    ax.set_yticklabels(labels)
    threshold = cm.max() / 2 if cm.size else 0
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            val = int(cm[i, j])
            color = "white" if val > threshold else "black"
            ax.text(j, i, str(val), ha="center", va="center", color=color)
    return im

# ============================================================
# 1) Suicide Risk Model: inline training/testing evaluation plots
# ============================================================
suicide_model, _ = load_model(MODELS_DIR / "suicide_risk_model.joblib")
s_texts, s_labels_raw = read_text_label_csv(PROCESSED_DIR / "suicide_test_split.csv")
s_true = np.array([normalize_suicide_label(x) for x in s_labels_raw], dtype=int)

if hasattr(suicide_model, "predict_proba"):
    prob_matrix = suicide_model.predict_proba(s_texts)
    class_names = [str(c).strip().lower() for c in getattr(suicide_model, "classes_", [])]
    if "suicide" in class_names:
        s_prob = prob_matrix[:, class_names.index("suicide")]
    elif prob_matrix.shape[1] >= 2:
        s_prob = prob_matrix[:, 1]
    else:
        s_prob = prob_matrix[:, 0]
else:
    raise RuntimeError("Suicide model has no predict_proba; cannot draw ROC/PR.")

s_pred = (s_prob >= 0.5).astype(int)
s_acc = accuracy_score(s_true, s_pred)
s_f1 = f1_score(s_true, s_pred, average="binary", zero_division=0)
s_cm = confusion_matrix(s_true, s_pred, labels=[0, 1])
fpr, tpr, _ = roc_curve(s_true, s_prob)
roc_auc = auc(fpr, tpr)
prec, rec, _ = precision_recall_curve(s_true, s_prob)
baseline = float(np.mean(s_true))

print(f"Suicide model -> Accuracy: {s_acc:.4f}, F1: {s_f1:.4f}, ROC-AUC: {roc_auc:.4f}")

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
im = draw_cm(axes[0, 0], s_cm, ["non-suicide", "suicide"], "Suicide Model - Confusion Matrix")
fig.colorbar(im, ax=axes[0, 0], fraction=0.046, pad=0.04)

axes[0, 1].plot(fpr, tpr, linewidth=2.2, label=f"ROC AUC={roc_auc:.4f}")
axes[0, 1].plot([0, 1], [0, 1], "--", color="gray", label="Random")
axes[0, 1].set_title("Suicide Model - ROC Curve")
axes[0, 1].set_xlabel("False Positive Rate")
axes[0, 1].set_ylabel("True Positive Rate")
axes[0, 1].legend(loc="lower right")

axes[1, 0].plot(rec, prec, linewidth=2.2, color="#2ca02c", label="PR Curve")
axes[1, 0].axhline(baseline, linestyle="--", color="gray", label=f"Baseline={baseline:.4f}")
axes[1, 0].set_title("Suicide Model - Precision/Recall")
axes[1, 0].set_xlabel("Recall")
axes[1, 0].set_ylabel("Precision")
axes[1, 0].legend(loc="lower left")

axes[1, 1].hist(s_prob[s_true == 0], bins=30, alpha=0.65, label="True non-suicide", color="#6baed6")
axes[1, 1].hist(s_prob[s_true == 1], bins=30, alpha=0.65, label="True suicide", color="#fb6a4a")
axes[1, 1].axvline(0.5, color="black", linestyle="--", label="Threshold 0.5")
axes[1, 1].set_title("Suicide Model - Probability Distribution")
axes[1, 1].set_xlabel("Predicted suicide probability")
axes[1, 1].set_ylabel("Count")
axes[1, 1].legend(loc="upper center")

plt.tight_layout()
fig.savefig(SAVE_DIR / "suicide_model_eval_inline.png", dpi=180)
plt.show()

# ============================================================
# 2) Journal Emotion + Sentiment Models
# ============================================================
emotion_model, _ = load_model(MODELS_DIR / "journal_emotion_model.joblib")
e_texts, e_true = read_text_label_csv(TEST_DIR / "journal_emotion_test.csv")
e_true = [x.strip().lower() for x in e_true]
e_pred = [str(x).strip().lower() for x in emotion_model.predict(e_texts)]
e_labels = sorted(set(e_true) | set(e_pred))
e_cm = confusion_matrix(e_true, e_pred, labels=e_labels)
e_acc = accuracy_score(e_true, e_pred)
e_f1 = f1_score(e_true, e_pred, average="macro", zero_division=0)

sent_model, _ = load_model(MODELS_DIR / "journal_sentiment_model.joblib")
t_texts, t_true = read_text_label_csv(TEST_DIR / "journal_sentiment_test.csv")
t_true = [x.strip().lower() for x in t_true]
t_pred = [str(x).strip().lower() for x in sent_model.predict(t_texts)]
t_labels = sorted(set(t_true) | set(t_pred))
t_cm = confusion_matrix(t_true, t_pred, labels=t_labels)
t_acc = accuracy_score(t_true, t_pred)
t_f1 = f1_score(t_true, t_pred, average="macro", zero_division=0)

print(f"Emotion model -> Accuracy: {e_acc:.4f}, Macro-F1: {e_f1:.4f}")
print(f"Sentiment model -> Accuracy: {t_acc:.4f}, Macro-F1: {t_f1:.4f}")

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
im1 = draw_cm(axes[0], e_cm, e_labels, "Journal Emotion Model - Confusion Matrix")
im2 = draw_cm(axes[1], t_cm, t_labels, "Journal Sentiment Model - Confusion Matrix")
fig.colorbar(im1, ax=axes[0], fraction=0.046, pad=0.04)
fig.colorbar(im2, ax=axes[1], fraction=0.046, pad=0.04)
plt.tight_layout()
fig.savefig(SAVE_DIR / "journal_models_eval_inline.png", dpi=180)
plt.show()

# ============================================================
# 3) Train/Test dataset distribution plots for requested CSV pairs
# ============================================================
dataset_specs = [
    ("mood_forecast", "next_day_mood_label", "Mood Forecast"),
    ("mood_pattern", "pattern_label", "Mood Pattern"),
    ("weekly_wellness", "weekly_label", "Weekly Wellness"),
    ("cosmic_insights", "sun_sign", "Cosmic Insights"),
    ("journal_emotion", "label", "Journal Emotion"),
    ("journal_sentiment", "label", "Journal Sentiment"),
]

for ds, label_col, title in dataset_specs:
    train_csv = TRAIN_DIR / f"{ds}_train.csv"
    test_csv = TEST_DIR / f"{ds}_test.csv"

    train_vals = read_col(train_csv, label_col)
    test_vals = read_col(test_csv, label_col)

    train_counter = Counter(train_vals)
    test_counter = Counter(test_vals)

    labels = sorted(set(train_counter.keys()) | set(test_counter.keys()))
    x = np.arange(len(labels))
    width = 0.38

    fig, ax = plt.subplots(figsize=(10, 5.5))
    bars1 = ax.bar(x - width / 2, [train_counter.get(k, 0) for k in labels], width=width, label="Train", color="#4c78a8")
    bars2 = ax.bar(x + width / 2, [test_counter.get(k, 0) for k in labels], width=width, label="Test", color="#f58518")

    ax.set_title(f"{title} - Train vs Test Label Distribution")
    ax.set_xlabel(label_col)
    ax.set_ylabel("Row count")
    ax.set_xticks(x)
    ax.set_xticklabels(labels, rotation=25, ha="right")
    ax.legend(loc="upper right")

    max_count = max([0] + [int(b.get_height()) for b in list(bars1) + list(bars2)])
    for bars in (bars1, bars2):
        for bar in bars:
            h = int(bar.get_height())
            ax.text(bar.get_x() + bar.get_width() / 2, h + max(1, int(max_count * 0.01)), str(h), ha="center", va="bottom", fontsize=8)

    plt.tight_layout()
    fig.savefig(SAVE_DIR / f"{ds}_train_test_distribution_inline.png", dpi=180)
    plt.show()

# Stats box numeric histograms
stats_train = TRAIN_DIR / "stats_box_train.csv"
stats_test = TEST_DIR / "stats_box_test.csv"
stats_fields = ["stress_index", "emotional_stability", "positivity_ratio", "confidence_score"]
stats_titles = {
    "stress_index": "Stress Index",
    "emotional_stability": "Emotional Stability",
    "positivity_ratio": "Positivity Ratio",
    "confidence_score": "Confidence Score",
}

fig, axes = plt.subplots(2, 2, figsize=(10.5, 7.2))
for idx, field in enumerate(stats_fields):
    ax = axes.flatten()[idx]
    tr = read_numeric_col(stats_train, field)
    te = read_numeric_col(stats_test, field)
    ax.hist(tr, bins=20, alpha=0.65, color="#4c78a8", label="Train")
    ax.hist(te, bins=20, alpha=0.65, color="#f58518", label="Test")
    ax.set_title(stats_titles[field])
    ax.set_xlabel("Value")
    ax.set_ylabel("Count")
    if idx == 0:
        ax.legend(loc="upper right")

fig.suptitle("Stats Box - Train vs Test Numeric Distributions", fontsize=13)
plt.tight_layout(rect=[0, 0, 1, 0.96])
fig.savefig(SAVE_DIR / "stats_box_train_test_histograms_inline.png", dpi=180)
plt.show()

# Row count overview across all requested datasets
overview_datasets = [
    "mood_forecast",
    "mood_pattern",
    "weekly_wellness",
    "stats_box",
    "cosmic_insights",
    "journal_emotion",
    "journal_sentiment",
]

train_counts, test_counts = [], []
for ds in overview_datasets:
    tr_csv = TRAIN_DIR / f"{ds}_train.csv"
    te_csv = TEST_DIR / f"{ds}_test.csv"
    tr_n = sum(1 for _ in csv.DictReader(tr_csv.open("r", encoding="utf-8", newline="", errors="ignore")))
    te_n = sum(1 for _ in csv.DictReader(te_csv.open("r", encoding="utf-8", newline="", errors="ignore")))
    train_counts.append(tr_n)
    test_counts.append(te_n)

x = np.arange(len(overview_datasets))
w = 0.38
fig, ax = plt.subplots(figsize=(11.2, 5.8))
ax.bar(x - w / 2, train_counts, width=w, label="Train", color="#4c78a8")
ax.bar(x + w / 2, test_counts, width=w, label="Test", color="#f58518")
ax.set_title("All Requested Datasets - Train vs Test Row Counts")
ax.set_xlabel("Dataset")
ax.set_ylabel("Rows")
ax.set_xticks(x)
ax.set_xticklabels(overview_datasets, rotation=25, ha="right")
ax.legend(loc="upper right")
plt.tight_layout()
fig.savefig(SAVE_DIR / "all_datasets_train_test_row_counts_inline.png", dpi=180)
plt.show()

print("\nGenerated inline notebook PNG files:")
for p in sorted(SAVE_DIR.glob("*.png")):
    print("-", p.name)
